# Lecture 9 — GPU Computing: Architecture, Programming Model & CUDA Kernels

**Numerical Scientific Computing — Aalborg University**

---

This tutorial accompanies Lecture 9.  
By the end of this notebook you will be able to:

- Query GPU hardware properties from Python
- Explain *why* GPUs outperform CPUs on data-parallel workloads
- Describe the GPU memory hierarchy and its bandwidth implications
- Understand the thread → warp → block → grid hierarchy
- Use **CuPy** for GPU-accelerated array operations without writing kernels
- Write, launch, and benchmark custom **Numba CUDA kernels**

> **Exercises 9.0–9.3 are assessed separately** and are NOT solved here.  
> This notebook gives you all the building blocks you need.

---

### Requirements

```
numpy  matplotlib  cupy  numba  numba-cuda
```

Run on **AI-LAB** (or any system with a CUDA-capable GPU).  
CPU-only cells are marked with 🖥️; GPU cells are marked with ⚡.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import numpy as np
import cupy as cp
import time
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from numba import cuda

# Dark theme consistent with lecture slides
plt.style.use('dark_background')
matplotlib.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'text.color':       '#C9D1D9',
    'grid.color':       '#21262D',
    'font.family':      'DejaVu Sans',
})

ACCENT   = '#4DD0E1'   # cyan  — registers / fast
PURPLE   = '#CE93D8'   # purple — shared mem
YELLOW   = '#FFF176'   # yellow — global mem
GREEN    = '#80CBC4'   # teal   — SM / device
ORANGE   = '#FFB74D'   # orange — warnings

print('All imports OK.')

---

## Section 1 — GPU Introspection ⚡

Before running any kernel it is useful to know what hardware you are working with.  
Every GPU exposes its capabilities through a **device object** that you can query at runtime.

Key properties to check:

| Property | Why it matters |
|---|---|
| **Compute Capability (CC)** | Determines which CUDA features are available (e.g. clusters require CC 9.0+) |
| **SM count** | Sets the maximum parallelism — more SMs = more blocks running concurrently |
| **Warp size** | Always 32 on NVIDIA GPUs; your block sizes should be multiples of this |
| **Shared mem / block** | Caps how much fast on-chip memory a single block can use (~48 KB typical) |
| **Max threads / block** | Upper limit on block size (1024 on modern GPUs) |
| **Global memory** | Total VRAM — the "RAM" of your GPU |

In [ ]:
# ── 1a. Print GPU device properties ───────────────────────────────────────
print('=' * 55)
for i, gpu in enumerate(cuda.gpus):
    with gpu:
        mem   = cuda.current_context().get_memory_info()
        dev   = cuda.get_current_device()
        name  = dev.name.decode() if isinstance(dev.name, bytes) else dev.name
        print(f'Device {i}  —  {name}')
        print(f'  Compute Capability  : {dev.compute_capability}')
        print(f'  Streaming MPs (SMs) : {dev.MULTIPROCESSOR_COUNT}')
        print(f'  Warp size           : {dev.WARP_SIZE}')
        print(f'  Max threads / block : {dev.MAX_THREADS_PER_BLOCK}')
        print(f'  Shared mem / block  : {dev.MAX_SHARED_MEMORY_PER_BLOCK / 1024:.0f} KB')
        print(f'  Constant memory     : {dev.TOTAL_CONSTANT_MEMORY / 1024:.0f} KB')
        print(f'  Global memory       : {mem.total / 2**30:.2f} GB')
        print(f'  Clock rate          : {dev.CLOCK_RATE / 1_000:.0f} MHz')
        print('=' * 55)

In [ ]:
# ── 1b. Visualise memory sizes across the hierarchy ────────────────────────
#
# We plot the log-scale size of each memory level so the differences are visible.

with cuda.gpus[0]:
    mem = cuda.current_context().get_memory_info()
    dev = cuda.get_current_device()
    smem_kb   = dev.MAX_SHARED_MEMORY_PER_BLOCK / 1024
    const_kb  = dev.TOTAL_CONSTANT_MEMORY / 1024
    global_mb = mem.total / 2**20

# Typical register file per SM ≈ 256 KB (not directly queryable — use typical value)
reg_kb = 256
l2_mb  = 40 * 1024   # ~40 MB typical (A100 80 MB); shown in KB for uniform axis

levels = ['Registers\n(per SM)', 'Shared Mem / L1\n(per SM)',
          'Constant\nCache', 'L2 Cache', 'Global Mem\n(VRAM)']
sizes_kb = [reg_kb, smem_kb, const_kb, l2_mb, global_mb * 1024]
colours  = [ACCENT, PURPLE, '#F9A825', '#7B1FA2', YELLOW]

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.barh(levels, sizes_kb, color=colours, edgecolor='#30363D', height=0.55)
ax.set_xscale('log')
ax.set_xlabel('Size (KB, log scale)')
ax.set_title('GPU Memory Hierarchy — Size per Level (read from device)', pad=12)
ax.axvline(1, color='#30363D', linewidth=0.6)

# Annotate with human-readable sizes
human = [f'{reg_kb} KB', f'{smem_kb:.0f} KB', f'{const_kb:.0f} KB',
         f'{l2_mb/1024:.0f} MB (est.)', f'{global_mb/1024:.1f} GB']
for bar, label in zip(bars, human):
    ax.text(bar.get_width() * 1.5, bar.get_y() + bar.get_height() / 2,
            label, va='center', ha='left', fontsize=9, color='#C9D1D9')

# Speed annotation arrows
ax.annotate('← fastest', xy=(sizes_kb[0], 0), xytext=(sizes_kb[0]*3, -0.6),
            fontsize=8, color=ACCENT,
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.2))
ax.annotate('slowest →', xy=(sizes_kb[4], 4), xytext=(sizes_kb[4]/200, 3.6),
            fontsize=8, color=YELLOW,
            arrowprops=dict(arrowstyle='->', color=YELLOW, lw=1.2))

ax.set_xlim(right=sizes_kb[-1] * 40)
plt.tight_layout()
plt.show()

---

## Section 2 — Why GPUs? CPU vs GPU Throughput 🖥️⚡

GPUs are optimised for **throughput** — executing the *same operation* on millions of data elements in parallel.  
CPUs are optimised for **latency** — executing a single instruction sequence as fast as possible with deep caches and branch prediction.

The fundamental difference:

```
CPU:  4–32 cores × very fast clock × large cache  → low latency per task
GPU:  thousands of small cores × modest clock     → enormous throughput
```

For data-parallel numerical work (element-wise array operations, matrix products) the GPU wins by a large margin once the array is large enough to fill all the cores.  
For small arrays, **PCIe transfer overhead** can actually make the GPU *slower* — that crossover point is critical to understand.

In [ ]:
# ── 2a. Measure CPU vs GPU throughput for element-wise sqrt + abs + sin ───

def benchmark(fn, *args, repeats=10, warmup=3):
    """Return median wall-clock time in seconds over `repeats` runs."""
    for _ in range(warmup):
        fn(*args)
    cp.cuda.Stream.null.synchronize()   # flush any GPU work
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        cp.cuda.Stream.null.synchronize()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


def cpu_op(a):
    return np.sqrt(np.abs(np.sin(a)))


def gpu_op(a_gpu):
    return cp.sqrt(cp.abs(cp.sin(a_gpu)))


sizes   = [2**k for k in range(10, 27, 2)]   # 1 K … 64 M elements
cpu_t, gpu_t, speedup = [], [], []

for N in sizes:
    a_cpu = np.random.rand(N).astype(np.float32)
    a_gpu = cp.asarray(a_cpu)

    tc = benchmark(cpu_op, a_cpu)
    tg = benchmark(gpu_op, a_gpu)

    cpu_t.append(tc * 1e3)
    gpu_t.append(tg * 1e3)
    speedup.append(tc / tg)
    print(f'N={N:>10,}  CPU {tc*1e3:7.2f} ms  GPU {tg*1e3:7.2f} ms  speedup {tc/tg:6.1f}×')

In [ ]:
# ── 2b. Plot the results ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
x = [N / 1e6 for N in sizes]   # millions of elements

# Left: absolute timings
ax1.loglog(x, cpu_t, 'o-', color='#90CAF9', lw=2, label='CPU (NumPy)')
ax1.loglog(x, gpu_t, 's-', color=ACCENT,    lw=2, label='GPU (CuPy)')
ax1.set_xlabel('Array size (million float32 elements)')
ax1.set_ylabel('Time (ms, log scale)')
ax1.set_title('sqrt(|sin(a)|)  —  wall-clock time')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)

# Right: speedup
ax2.semilogx(x, speedup, 'D-', color=GREEN, lw=2)
ax2.axhline(1, color=ORANGE, linestyle='--', lw=1.2, label='break-even (1×)')
ax2.fill_between(x, 1, speedup,
                 where=[s > 1 for s in speedup],
                 alpha=0.15, color=GREEN, label='GPU faster')
ax2.fill_between(x, speedup, 1,
                 where=[s < 1 for s in speedup],
                 alpha=0.15, color=ORANGE, label='CPU faster')
ax2.set_xlabel('Array size (million float32 elements)')
ax2.set_ylabel('Speedup (CPU time / GPU time)')
ax2.set_title('GPU speedup over CPU')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)

plt.suptitle('Element-wise  sqrt(|sin(a)|)  on float32 arrays', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

### 🔍 What to observe

- For **small arrays** the GPU may be slower — the kernel launch overhead dominates.
- Beyond a **crossover point** (~millions of elements) the GPU achieves significant speedup.
- The speedup **plateaus** once the GPU is fully saturated (all SMs busy).

---

### 🧑‍💻 Student Task 1 — Benchmark a Chain of Operations

The demo above benchmarks `sqrt(|sin(a)|)`.  
Your task is to benchmark a more expensive chain:

$$f(x) = \exp\!\left(-x^2\right) \cdot \sin(3x) + \log\!\left(|x| + 0.01\right)$$

**Steps:**
1. Implement `cpu_task1(a)` using NumPy.
2. Implement `gpu_task1(a_gpu)` using CuPy.
3. Measure times for array sizes from 10 K to 100 M elements and plot speedup vs N.
4. At which array size does the GPU become faster?  Write your answer in a markdown cell below.

*Starter code is provided — it runs as-is and produces a placeholder plot. Fill in the real implementations.*

In [ ]:
# ── Student Task 1 — starter code (runs as-is) ───────────────────────────

def cpu_task1(a):
    # TODO: implement  f(x) = exp(-x²) * sin(3x) + log(|x| + 0.01)
    return np.zeros_like(a)   # placeholder — replace this line


def gpu_task1(a_gpu):
    # TODO: implement the same formula with CuPy
    return cp.zeros_like(a_gpu)   # placeholder — replace this line


# ── Benchmark ────────────────────────────────────────────────────────────
task1_sizes = [int(10**k) for k in np.linspace(4, 8, 10)]
task1_cpu_t, task1_gpu_t, task1_speedup = [], [], []

for N in task1_sizes:
    a_cpu = np.random.rand(N).astype(np.float32) * 4 - 2   # range [-2, 2]
    a_gpu = cp.asarray(a_cpu)

    tc = benchmark(cpu_task1, a_cpu)
    tg = benchmark(gpu_task1, a_gpu)

    task1_cpu_t.append(tc * 1e3)
    task1_gpu_t.append(tg * 1e3)
    task1_speedup.append(tc / tg)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(task1_sizes, task1_speedup, 'o-', color=PURPLE, lw=2)
ax.axhline(1, color=ORANGE, linestyle='--', lw=1.2, label='break-even')
ax.set_xlabel('Array size (elements)')
ax.set_ylabel('Speedup (CPU / GPU)')
ax.set_title('Task 1 — GPU speedup for f(x) chain')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()
print('Placeholder outputs — implement cpu_task1 and gpu_task1 above.')

---

## Section 3 — GPU Memory Hierarchy ⚡

Understanding memory is *the* key to writing fast GPU code.  
The hierarchy from fastest to slowest:

```
Registers  ──▶  Shared Memory / L1  ──▶  Constant Cache  ──▶  L2  ──▶  Global (VRAM)
  ~1 cycle         ~5–10 cycles            ~10 cycles       ~30–50      ~200–600
  private/thread   per block 48 KB         read-only 64 KB  all SMs     entire GPU
```

### Transfer Bottleneck

Data must travel from **System RAM** to **VRAM** before the GPU can touch it.  
This happens over the **PCIe bus** at ~32 GB/s — roughly 60× slower than VRAM's internal bandwidth (~2 TB/s).  
For a computation to benefit from GPU acceleration, the compute time must outweigh the transfer time.

> **Rule of thumb:** if your kernel does only one or two operations per element, transfer overhead will likely dominate.  
> If your kernel is doing $O(N^2)$ or $O(N \log N)$ work on $N$ elements, the GPU almost always wins.

In [ ]:
# ── 3a. Measure Host→Device and Device→Host bandwidth ─────────────────────

transfer_sizes  = [2**k for k in range(16, 29, 2)]   # 64 KB … 512 MB
h2d_bw, d2h_bw = [], []

for nbytes in transfer_sizes:
    N = nbytes // 4   # float32
    h = np.random.rand(N).astype(np.float32)

    # Host → Device
    reps = 5
    start = cp.cuda.Event()
    end   = cp.cuda.Event()
    start.record()
    for _ in range(reps):
        d = cp.asarray(h)
    end.record()
    end.synchronize()
    t_h2d = cp.cuda.get_elapsed_time(start, end) / 1e3 / reps   # seconds
    h2d_bw.append(nbytes / t_h2d / 2**30)   # GB/s

    # Device → Host
    start.record()
    for _ in range(reps):
        _ = cp.asnumpy(d)
    end.record()
    end.synchronize()
    t_d2h = cp.cuda.get_elapsed_time(start, end) / 1e3 / reps
    d2h_bw.append(nbytes / t_d2h / 2**30)

x_mb = [s / 2**20 for s in transfer_sizes]

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(x_mb, h2d_bw, 'o-', color=ORANGE,  lw=2, label='Host → Device')
ax.semilogx(x_mb, d2h_bw, 's-', color='#90CAF9', lw=2, label='Device → Host')
ax.axhline(32, color='#C62828', linestyle='--', lw=1.2,
           label='PCIe 4.0 x16 theoretical peak (~32 GB/s)')
ax.set_xlabel('Transfer size (MB, log scale)')
ax.set_ylabel('Bandwidth (GB/s)')
ax.set_title('PCIe Transfer Bandwidth (Host ↔ Device)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Peak H→D: {max(h2d_bw):.1f} GB/s')
print(f'Peak D→H: {max(d2h_bw):.1f} GB/s')

---

## Section 4 — Thread Hierarchy Visualised 🖥️

CUDA organises parallel threads into a 3-level hierarchy:

| Level | Size | Shares |
|---|---|---|
| **Thread** | 1 CUDA Core | Registers (private) |
| **Warp** (32 threads) | 32 cores on 1 SM | — |
| **Thread Block** | up to 1024 threads → 1 SM | Shared Memory / L1 |
| **Grid** | all blocks → all SMs | Global Memory (VRAM) |

### SIMT — Single Instruction, Multiple Threads

Threads within the same **warp** execute the **same instruction simultaneously**.  
If threads in a warp take different branches (`if/else`), the warp must **serialise** the branches — one branch executes while the other lanes are **masked** (idle).  
This is called **warp divergence** and reduces effective throughput.

```python
# BAD — half the warp is idle in each branch
if threadIdx.x % 2 == 0:
    do_heavy_work()   # even lanes active, odd lanes masked
else:
    do_light_work()   # odd lanes active, even lanes masked

# GOOD — all threads in each warp take the same branch
# (achieved by structuring data so adjacent threads have similar control flow)
```

**Best practice:** make your block size a **multiple of 32**.

In [ ]:
# ── 4a. Visualise a 1D grid of thread blocks ─────────────────────────────

N            = 100    # total problem size (elements)
threads_per_block = 32
blocks_per_grid  = (N + threads_per_block - 1) // threads_per_block

fig, ax = plt.subplots(figsize=(14, 2.8))
ax.set_xlim(-0.5, N - 0.5)
ax.set_ylim(-0.5, 2.2)
ax.axis('off')
ax.set_title(f'1D Grid: N={N} elements → {blocks_per_grid} blocks × {threads_per_block} threads/block',
             fontsize=11, pad=8)

colors_block = plt.cm.tab20.colors

for i in range(N):
    block_id = i // threads_per_block
    active   = (i < N)
    fc = colors_block[block_id % len(colors_block)] if active else '#1F2937'
    rect = mpatches.FancyBboxPatch(
        (i - 0.45, 0.6), 0.9, 1.0,
        boxstyle='round,pad=0.05',
        facecolor=fc, edgecolor='#30363D', linewidth=0.6, alpha=0.85)
    ax.add_patch(rect)
    ax.text(i, 1.12, str(i), ha='center', va='center',
            fontsize=5.5, color='white')

# Block labels
for b in range(blocks_per_grid):
    mid = b * threads_per_block + threads_per_block / 2 - 0.5
    ax.text(mid, 0.3, f'Block {b}', ha='center', va='center',
            fontsize=8, color=colors_block[b % len(colors_block)])
    # bracket
    lo = b * threads_per_block - 0.4
    hi = min((b + 1) * threads_per_block, N) - 0.6
    ax.plot([lo, lo, hi, hi], [0.45, 0.42, 0.42, 0.45],
            color=colors_block[b % len(colors_block)], lw=1.2)

ax.text(N / 2, 2.05, f'Grid (1D)', ha='center', fontsize=9, color='#8B949E',
        style='italic')
plt.tight_layout()
plt.show()
print(f'Grid launch: kernel<<<{blocks_per_grid}, {threads_per_block}>>>  '
      f'({blocks_per_grid * threads_per_block} threads, covers {N} elements)')

In [ ]:
# ── 4b. Visualise warp divergence in a 32-thread warp ────────────────────

warp_size = 32
fig, axes = plt.subplots(1, 2, figsize=(11, 2.6))

for ax_idx, (branch_fn, title) in enumerate([
    (lambda i: i % 2 == 0,        'Divergent warp (even/odd split)'),
    (lambda i: i < warp_size // 2, 'No divergence (first 16 vs last 16)'),
]):
    ax = axes[ax_idx]
    ax.set_xlim(-0.5, warp_size - 0.5)
    ax.set_ylim(-0.3, 1.6)
    ax.axis('off')
    ax.set_title(title, fontsize=9)

    for i in range(warp_size):
        branch_A = branch_fn(i)
        fc = ACCENT if branch_A else '#1A3A2E'
        ec = '#26A69A' if not branch_A else ACCENT
        rect = mpatches.FancyBboxPatch(
            (i - 0.42, 0.3), 0.84, 0.9,
            boxstyle='round,pad=0.04',
            facecolor=fc, edgecolor=ec, linewidth=0.8)
        ax.add_patch(rect)
        ax.text(i, 0.75, str(i), ha='center', va='center',
                fontsize=5.5, color='white')

    # Use Patch only as legend handles -- never add_patch an abstract Patch
    legend_handles = [
        mpatches.Patch(facecolor=ACCENT,    label='Branch A (active)'),
        mpatches.Patch(facecolor='#1A3A2E', edgecolor='#26A69A',
                       label='Branch B (masked during A)'),
    ]
    ax.legend(handles=legend_handles, loc='upper right',
              fontsize=7, framealpha=0.4)

plt.suptitle('Warp Divergence Visualisation (32-lane warp)', y=1.02, fontsize=10)
plt.tight_layout()
plt.show()

---

## Section 5 — CuPy: GPU Arrays Without Kernels ⚡

**CuPy** is a drop-in NumPy/SciPy replacement that runs on the GPU.  
You swap `import numpy as np` for `import cupy as cp` and most code just works.

```python
import cupy as cp

a = cp.random.rand(1_000_000, dtype=cp.float32)   # array lives in VRAM
b = cp.sin(a) + cp.cos(a)                         # runs GPU kernels automatically
result = float(cp.sum(b))                         # scalar pulled back to CPU
```

CuPy automatically handles:
- Grid and block configuration
- Memory allocation on the GPU
- Kernel compilation (cached after first run)

It is ideal when the NumPy API covers your algorithm.  
When you need fine-grained control (custom memory access patterns, shared memory tiling) you need a **custom kernel** — covered in Section 7.

In [ ]:
# ── 5a. Matrix multiplication benchmark ───────────────────────────────────
#
# Matrix multiply is O(N³) compute but only O(N²) data — an ideal GPU workload.

matmul_sizes = [64, 128, 256, 512, 1024, 2048, 4096]
matmul_cpu, matmul_gpu = [], []

for n in matmul_sizes:
    A_cpu = np.random.rand(n, n).astype(np.float32)
    B_cpu = np.random.rand(n, n).astype(np.float32)
    A_gpu = cp.asarray(A_cpu)
    B_gpu = cp.asarray(B_cpu)

    tc = benchmark(np.matmul, A_cpu, B_cpu, repeats=5, warmup=2)
    tg = benchmark(cp.matmul, A_gpu, B_gpu, repeats=10, warmup=5)

    matmul_cpu.append(tc * 1e3)
    matmul_gpu.append(tg * 1e3)
    flops = 2 * n**3 / tg / 1e12   # TFLOP/s
    print(f'n={n:5d}  CPU {tc*1e3:8.2f} ms  GPU {tg*1e3:8.3f} ms  speedup {tc/tg:7.1f}×  '
          f'GPU TFLOP/s {flops:.2f}')

In [ ]:
# ── 5b. Plot matmul speedup ────────────────────────────────────────────────
speedup_mm = [c / g for c, g in zip(matmul_cpu, matmul_gpu)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.loglog(matmul_sizes, matmul_cpu, 'o-', color='#90CAF9', lw=2, label='CPU (NumPy)')
ax1.loglog(matmul_sizes, matmul_gpu, 's-', color=ACCENT,    lw=2, label='GPU (CuPy)')
ax1.set_xlabel('Matrix size n (n×n)')
ax1.set_ylabel('Time (ms, log)')
ax1.set_title('Matrix Multiply A @ B')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)

ax2.plot(matmul_sizes, speedup_mm, 'D-', color=GREEN, lw=2)
ax2.axhline(1, color=ORANGE, linestyle='--', lw=1.2)
ax2.set_xlabel('Matrix size n (n×n)')
ax2.set_ylabel('Speedup')
ax2.set_title('GPU speedup — matrix multiply')
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

### 🧑‍💻 Student Task 2 — Polynomial Evaluation with CuPy

Evaluate the polynomial:

$$p(x) = 3x^5 - 2x^4 + x^3 - 7x^2 + 4x - 1$$

on a float32 array of size $N$ using **Horner's method** for numerical stability:

$$p(x) = ((((3x - 2)x + 1)x - 7)x + 4)x - 1$$

**Steps:**
1. Implement `cpu_poly(x)` using NumPy and Horner's method.
2. Implement `gpu_poly(x_gpu)` using CuPy and Horner's method.
3. Verify both give the same result (use `np.allclose`).
4. Benchmark both for N from 10 K to 50 M and plot speedup vs N.
5. Is the GPU speedup higher or lower than for `sqrt(|sin(a)|)` from Section 2?  Why?

*Starter code runs as-is — fill in the real implementations.*

In [ ]:
# ── Student Task 2 — starter code (runs as-is) ───────────────────────────

def cpu_poly(x):
    # TODO: implement Horner's method for p(x) = 3x⁵ - 2x⁴ + x³ - 7x² + 4x - 1
    return np.zeros_like(x)   # placeholder


def gpu_poly(x_gpu):
    # TODO: same with CuPy
    return cp.zeros_like(x_gpu)   # placeholder


# Quick verification (will pass trivially with placeholders — fix after implementing)
x_test     = np.array([0.0, 1.0, -1.0, 2.0], dtype=np.float32)
x_test_gpu = cp.asarray(x_test)
cpu_res = cpu_poly(x_test)
gpu_res = cp.asnumpy(gpu_poly(x_test_gpu))
print('Verification (will fail after correct implementation if results differ):')
print('  cpu:', cpu_res)
print('  gpu:', gpu_res)
print('  match:', np.allclose(cpu_res, gpu_res, atol=1e-4))


# Benchmark
poly_sizes   = [int(10**k) for k in np.linspace(4, 7.7, 10)]
poly_cpu_t, poly_gpu_t, poly_speedup = [], [], []

for N in poly_sizes:
    xc = np.random.rand(N).astype(np.float32) * 4 - 2
    xg = cp.asarray(xc)

    tc = benchmark(cpu_poly, xc)
    tg = benchmark(gpu_poly, xg)
    poly_cpu_t.append(tc * 1e3)
    poly_gpu_t.append(tg * 1e3)
    poly_speedup.append(tc / tg)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(poly_sizes, poly_speedup, 'o-', color=PURPLE, lw=2, label='Task 2 poly')
ax.axhline(1, color=ORANGE, linestyle='--', lw=1.2, label='break-even')
ax.set_xlabel('Array size (elements)')
ax.set_ylabel('Speedup (CPU / GPU)')
ax.set_title('Task 2 — Polynomial Evaluation Speedup')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Section 6 — The GPU Programming Principle 🖥️

When writing GPU code you must **change your mental model**.

### CPU mindset — you write the loop:

```python
result = np.zeros(N)
for i in range(N):
    result[i] = a[i] + b[i]
```

One instruction stream processes all N elements **sequentially**.

### GPU mindset — write for **one element**:

```python
@cuda.jit
def add_kernel(a, b, result):
    i = cuda.grid(1)          # which element am I?
    if i < a.size:            # guard against out-of-bounds
        result[i] = a[i] + b[i]

add_kernel[blocks, threads](d_a, d_b, d_result)  # launch N copies in parallel
```

The kernel body is written for a **single thread** — the GPU runtime instantiates it across all N threads simultaneously.

**`cuda.grid(1)`** returns the global thread index:

```
i = blockIdx.x * blockDim.x + threadIdx.x
```

For 2D problems use `cuda.grid(2)` which returns `(row, col)`.

### Grid/Block sizing formula

```python
threads_per_block = 256        # must be multiple of 32, usually 128–512
blocks_per_grid   = (N + threads_per_block - 1) // threads_per_block
kernel[blocks_per_grid, threads_per_block](...)
```

The ceiling division ensures we have enough threads even when N is not a multiple of the block size.  
The `if i < a.size` guard prevents out-of-bounds access in the last block.

---

## Section 7 — Numba CUDA: Your First Custom Kernel ⚡

Numba CUDA lets you write GPU kernels in pure Python using the `@cuda.jit` decorator.  
The decorator JIT-compiles the function to PTX (NVIDIA assembly) the first time it is called.

**Key API:**

| Call | Meaning |
|---|---|
| `cuda.to_device(a)` | Copy NumPy array to GPU |
| `cuda.device_array(shape, dtype)` | Allocate uninitialised GPU array |
| `arr.copy_to_host()` | Copy GPU array back to NumPy |
| `cuda.grid(1)` | Global thread index (1D) |
| `cuda.grid(2)` | Global thread indices (2D) as `(row, col)` |
| `cuda.syncthreads()` | Barrier: all threads in a block wait here |

> **Package note:** the CUDA target built into Numba is deprecated.  
> Install and use `numba-cuda` for future-proof code:
> ```
> pip install numba-cuda
> ```
> The API is identical — only the package name changes.

In [ ]:
# ── 7a. Vector addition kernel ─────────────────────────────────────────────

@cuda.jit
def vector_add(a, b, out):
    """Compute out[i] = a[i] + b[i] for the thread's element."""
    i = cuda.grid(1)      # global 1D thread index
    if i < out.size:      # guard — last block may have spare threads
        out[i] = a[i] + b[i]


N = 1_000_000
h_a = np.random.rand(N).astype(np.float32)
h_b = np.random.rand(N).astype(np.float32)

# Transfer to GPU
d_a   = cuda.to_device(h_a)
d_b   = cuda.to_device(h_b)
d_out = cuda.device_array(N, dtype=np.float32)

# Launch configuration
TPB    = 256                          # threads per block
blocks = (N + TPB - 1) // TPB        # ceiling division

# Warm-up (trigger JIT compilation)
vector_add[blocks, TPB](d_a, d_b, d_out)
cuda.synchronize()

# Time the kernel
t0 = time.perf_counter()
for _ in range(20):
    vector_add[blocks, TPB](d_a, d_b, d_out)
cuda.synchronize()
t_kernel = (time.perf_counter() - t0) / 20 * 1e3

# Verify
h_out = d_out.copy_to_host()
expected = h_a + h_b
ok = np.allclose(h_out, expected, atol=1e-5)

print(f'N = {N:,}   blocks={blocks}   threads/block={TPB}')
print(f'Kernel time: {t_kernel:.3f} ms   Correct: {ok}')

In [ ]:
# ── 7b. Benchmark Numba kernel vs NumPy vs CuPy ───────────────────────────

sizes_bench  = [2**k for k in range(12, 26, 2)]
t_numpy, t_numba, t_cupy = [], [], []

for N in sizes_bench:
    ha = np.random.rand(N).astype(np.float32)
    hb = np.random.rand(N).astype(np.float32)
    da = cuda.to_device(ha)
    db = cuda.to_device(hb)
    do = cuda.device_array(N, dtype=np.float32)
    ca = cp.asarray(ha)
    cb = cp.asarray(hb)
    tpb   = 256
    blks  = (N + tpb - 1) // tpb

    def _numba(): vector_add[blks, tpb](da, db, do); cuda.synchronize()
    def _cupy():  _ = ca + cb; cp.cuda.Stream.null.synchronize()

    t_numpy.append(benchmark(np.add, ha, hb) * 1e3)
    t_numba.append(benchmark(_numba, repeats=20, warmup=3) * 1e3)
    t_cupy.append( benchmark(_cupy,  repeats=20, warmup=3) * 1e3)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = [s / 1e6 for s in sizes_bench]
ax.loglog(x, t_numpy, 'o-', color='#90CAF9', lw=2, label='NumPy (CPU)')
ax.loglog(x, t_cupy,  's-', color=ACCENT,    lw=2, label='CuPy (GPU)')
ax.loglog(x, t_numba, '^-', color=GREEN,     lw=2, label='Numba CUDA kernel (GPU)')
ax.set_xlabel('Array size (M float32 elements)')
ax.set_ylabel('Time (ms, log scale)')
ax.set_title('Vector Addition: NumPy vs CuPy vs Numba CUDA kernel')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Section 8 — A More Complex Kernel: ReLU Activation ⚡

The **ReLU** (Rectified Linear Unit) activation function is ubiquitous in deep learning:

$$\text{ReLU}(x) = \max(0, x)$$

Element-wise operations like this are trivial to parallelise.  
Here we also add a **leaky variant**:

$$\text{LeakyReLU}(x) = \begin{cases} x & x \ge 0 \\ \alpha x & x < 0 \end{cases}$$

This kernel involves a branch — but because adjacent elements in a real-world activation map tend to have the same sign (positive or negative regions), warp divergence is minimal in practice.

In [ ]:
# ── 8a. ReLU and Leaky-ReLU kernels ───────────────────────────────────────

@cuda.jit
def relu_kernel(x, out):
    """out[i] = max(0, x[i])"""
    i = cuda.grid(1)
    if i < x.size:
        out[i] = x[i] if x[i] > 0.0 else 0.0


@cuda.jit
def leaky_relu_kernel(x, out, alpha):
    """out[i] = x[i] if x[i] >= 0 else alpha * x[i]"""
    i = cuda.grid(1)
    if i < x.size:
        out[i] = x[i] if x[i] >= 0.0 else alpha * x[i]


# Generate a smooth signal spanning negatives and positives
N_sig = 2000
t_sig = np.linspace(-3, 3, N_sig, dtype=np.float32)
signal = (np.sin(3 * t_sig) + 0.5 * np.sin(7 * t_sig)).astype(np.float32)

d_sig  = cuda.to_device(signal)
d_relu = cuda.device_array(N_sig, dtype=np.float32)
d_lrlu = cuda.device_array(N_sig, dtype=np.float32)

TPB  = 128
blks = (N_sig + TPB - 1) // TPB

relu_kernel[blks, TPB](d_sig, d_relu)
leaky_relu_kernel[blks, TPB](d_sig, d_lrlu, np.float32(0.1))
cuda.synchronize()

h_relu = d_relu.copy_to_host()
h_lrlu = d_lrlu.copy_to_host()

# Plot
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, y, title, color in zip(
        axes,
        [signal, h_relu, h_lrlu],
        ['Original signal', 'ReLU output', 'Leaky ReLU (α=0.1)'],
        ['#90CAF9', ACCENT, PURPLE]):
    ax.plot(t_sig, y, color=color, lw=1.5)
    ax.axhline(0, color='#30363D', lw=0.8)
    ax.fill_between(t_sig, y, 0, where=(y > 0), alpha=0.2, color=color)
    ax.fill_between(t_sig, y, 0, where=(y < 0), alpha=0.15, color=ORANGE)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('t')
    ax.grid(alpha=0.2)

plt.suptitle('ReLU & Leaky ReLU — Numba CUDA kernels on a sine sum signal',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

print('ReLU   max neg value:', h_relu.min(), '  (should be 0.0)')
print('Leaky max neg value: ', h_lrlu.min())

### 🧑‍💻 Student Task 3 — 1D Averaging Stencil Kernel

Many numerical algorithms require **stencil operations** — each output element depends on neighbouring input elements.  
A classic example is a **1D weighted average (smoothing filter)**:

$$\text{out}[i] = 0.25 \cdot a[i-1] \;+\; 0.50 \cdot a[i] \;+\; 0.25 \cdot a[i+1]$$

Boundary condition: if $i = 0$ or $i = N-1$, treat out-of-bounds neighbours as 0.

**Steps:**
1. Implement `stencil_kernel(a, out)` using `@cuda.jit`.
   - Use `cuda.grid(1)` to get the global index.
   - Handle boundary threads correctly.
2. Test on a noisy sine wave (provided below) and visualise input vs output.
3. Run the kernel 3 times in sequence on the *same* output array.  
   What happens to the signal after multiple passes?  
   Add a markdown cell with your explanation.

*Starter code runs as-is — it applies the identity (copies input to output). Fill in the real stencil.*

In [ ]:
# ── Student Task 3 — starter code (runs as-is) ───────────────────────────

@cuda.jit
def stencil_kernel(a, out):
    """TODO: implement out[i] = 0.25*a[i-1] + 0.50*a[i] + 0.25*a[i+1]
    with boundary condition: treat out-of-bounds elements as 0.
    """
    i = cuda.grid(1)
    if i < out.size:
        # Placeholder: identity — replace with the stencil formula
        out[i] = a[i]


# ── Test signal: sine + noise ─────────────────────────────────────────────
N_stencil = 512
np.random.seed(42)
x_noisy = (np.sin(2 * np.pi * np.linspace(0, 4, N_stencil)) +
           0.4 * np.random.randn(N_stencil)).astype(np.float32)

d_in   = cuda.to_device(x_noisy)
d_out1 = cuda.device_array(N_stencil, dtype=np.float32)
d_out3 = cuda.device_array(N_stencil, dtype=np.float32)

TPB  = 128
blks = (N_stencil + TPB - 1) // TPB

# Single pass
stencil_kernel[blks, TPB](d_in, d_out1)
cuda.synchronize()
h_out1 = d_out1.copy_to_host()

# Three passes (chain: in → out3 → out3 → out3)
d_tmp = cuda.device_array(N_stencil, dtype=np.float32)
stencil_kernel[blks, TPB](d_in,   d_out3);  cuda.synchronize()
stencil_kernel[blks, TPB](d_out3, d_tmp);   cuda.synchronize()
stencil_kernel[blks, TPB](d_tmp,  d_out3);  cuda.synchronize()
h_out3 = d_out3.copy_to_host()

# Plot
idx = np.arange(N_stencil)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(idx, x_noisy, color='#546E7A', lw=1.2, alpha=0.7, label='Input (noisy)')
ax.plot(idx, h_out1,  color=ACCENT,    lw=1.8, label='1 stencil pass')
ax.plot(idx, h_out3,  color=ORANGE,    lw=1.8, label='3 stencil passes')
ax.set_xlabel('Index')
ax.set_ylabel('Amplitude')
ax.set_title('Task 3 — 1D Averaging Stencil Kernel  (placeholder = identity)')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print('Implement stencil_kernel above and re-run to see the smoothing effect.')

---

## Summary

| Topic | Key takeaway |
|---|---|
| **GPU architecture** | Thousands of small cores organised into SMs; optimised for throughput not latency |
| **Memory hierarchy** | Registers (fastest, private) → Shared/L1 → Constant → L2 → Global (VRAM, largest) |
| **Transfer bottleneck** | PCIe ~32 GB/s ≪ VRAM ~2 TB/s — minimise transfers, maximise reuse |
| **Thread hierarchy** | Thread → Warp (32) → Block (1 SM) → Grid (all SMs) |
| **SIMT & divergence** | All threads in a warp execute the same instruction; branches must be serialised |
| **CuPy** | Drop-in NumPy replacement for GPU arrays; automatic kernel launch |
| **Numba CUDA kernel** | `@cuda.jit`, write for ONE element, use `cuda.grid(1)`, guard with `if i < size` |
| **Grid sizing** | `blocks = ceil(N / threads_per_block)`, use multiple of 32 for block size |

---

### What's next?

**Lecture 10** will go deeper into GPU kernel optimisation:

- Shared memory tiling for matrix multiplication
- Memory coalescing — accessing global memory efficiently
- Occupancy and register pressure
- Profiling GPU kernels with `nsys` / `ncu`

---

*Good luck with Exercises 9.0–9.3!*